# 🔒 SIREN 用于真实 PII 数据：方法是否有效？能否分出 PII 类别？

把论文 **《LLM Safety From Within》**（arXiv:2604.18519）的方法**原样**用到真实 PII 数据上，回答两个问题：

1. **论文的方法对 PII 任务有效吗？** —— 二分类：这段文本含不含某类 PII
2. **能否更进一步，分出具体是哪一类 PII？** —— 多分类：SSN / 邮箱 / 电话 / 信用卡 / …

数据集：[`ai4privacy/pii-masking-200k`](https://huggingface.co/datasets/ai4privacy/pii-masking-200k) —— 真实自然语句 + 字符级 PII 标注（`{value, start, end, label}`），覆盖 40+ 个类别。

---

### ⚠️ 关键实验设计：正负样本都取自同一语料

这个数据集里**所有文本都含 PII，没有天然的负样本**。若拿通用语料充当负样本，就会重蹈本项目早期的覆辙：

> 早期版本用手写模板，正样本 `"SSN is 492-10-4921"`、负样本 `"Order ID is 492-10-4921"`，结果**第 1 层探针就拿到 F1 = 1.000**。因为单个判别词在词嵌入层就能分开两类，探针**根本没机会去读隐层**。

所以两个实验的正负样本**全部来自同一语料**：

| 实验 | 正样本 | 负样本 | 为什么没有捷径 |
|---|---|---|---|
| 二分类 | 含目标类 PII | 含**其他类** PII | 同一生成器、同一文风，只差标识符类型 |
| 多分类 | 判断属于哪一类 | — | 全部同源，区分类别必须真的理解标识符语义 |

**多分类是更强的证据**：领域、文风、句式全都一样，模型无法靠"这看起来像不像敏感场景"蒙对，只能真的表征出"这是社保号还是信用卡号"。

### 方法保持与论文一致
mean-pooling（式 2）· train/val/test 三分 · C 网格 `{100,200,500,1000}` **在验证集上选** · η=0.8 筛安全神经元 · α_l 由验证集性能算 · 报 **Macro-F1**

多分类下的唯一改动：`liblinear` 不支持 3 类以上，故用 **one-vs-rest** 包装（每类仍是 L1 + liblinear，与论文单类设定一致）；神经元筛选取**跨类别的最大权重**。

### 每次运行都带打乱标签对照
把训练标签打乱后重跑逐层探针。若真标签曲线是真信号，对照必须塌到**随机基线**（二分类 0.5，n 分类 1/n）。若两者都高，说明是维度过拟合而非真信号。

## 步骤 1：环境检查、克隆仓库与安装依赖

In [ ]:
!nvidia-smi

# 克隆仓库，使本 Notebook 可 `import siren`（单一真源；算法逻辑不内联）
import os
if not os.path.isdir('siren-pii-probing'):
    !git clone -q https://github.com/jackyluo-learning/siren-pii-probing.git
%cd siren-pii-probing

!pip install --quiet torch transformers scikit-learn matplotlib datasets tqdm scipy

## 步骤 2：先摸清数据集里有哪些 PII 类别

在选定实验类别之前，先扫一遍数据集，看各类别的出现频次。类别太稀疏的没法训练，需要按频次挑。

In [ ]:
!PYTHONPATH=.:examples python -u examples/run_pii_layerwise.py --task survey --cap 4000

## 实验 1：二分类 —— 论文的方法对 PII 有效吗？

任务：**这段文本含不含 `SOCIALNUM`（社保号）？**

- 正样本：含社保号的文本
- 负样本：**同一语料中含其他 PII 但不含社保号**的文本

负样本不是普通文本，而是同样谈论个人信息、同样带标识符的句子——所以探针不能靠"这段像不像敏感内容"取巧，必须分辨**标识符的类型**。

想换类别就改 `--target`（如 `EMAIL`、`CREDITCARDNUMBER`）。

In [ ]:
!PYTHONPATH=.:examples python -u examples/run_pii_layerwise.py \
    --task binary --target SOCIALNUM \
    --model "Qwen/Qwen3-4B" --cap 8000 --max-length 128

from IPython.display import Image, display
display(Image('pii_binary_socialnum_layers.png'))

## 实验 2：多分类 —— 能分出具体是哪一类 PII 吗？

任务：**这段文本携带的是哪一类 PII？**

只保留**恰好含一个**目标类别的文本，让标签无歧义；同时含多个目标类别的文本会被丢弃（脚本会报告丢弃数量，代价是可见的）。

这是比实验 1 更强的检验：所有样本来自同一语料、同一文风、同一生成器，**唯一的差别就是标识符的种类**。要做对，模型的内部表征必须真的把"社保号"和"信用卡号"编码成不同的东西。

产出两张图：**层级曲线**（看哪一层最能区分类别）与**混淆矩阵**（看哪些类别之间容易混淆）。

In [ ]:
# 不指定 --categories 则按频次自动取前 N 个类别；也可显式指定，例如：
#   --categories "SOCIALNUM,EMAIL,TELEPHONENUM,CREDITCARDNUMBER,DATEOFBIRTH,IPV4"
!PYTHONPATH=.:examples python -u examples/run_pii_layerwise.py \
    --task multiclass --n-categories 6 \
    --model "Qwen/Qwen3-4B" --cap 20000 --min-per-class 80 --max-length 128

from IPython.display import Image, display
display(Image('pii_multiclass_6way_layers.png'))
display(Image('pii_multiclass_6way_confusion.png'))